# Repeated-trials model-selection summary

Runs repeated-trial model selection for SPPCS, Levi, and CS, saves repeated-trial summaries, and exports website-ready repeated-trial tables.

In [1]:
from pathlib import Path
import sys
import json
import shutil
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "src").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

ARTIFACTS = PROJECT_ROOT / "artifacts"
WEBSITE_ROOT = PROJECT_ROOT / "docs" / "assets" / "model_selection"

os.chdir(PROJECT_ROOT)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("ARTIFACTS:", ARTIFACTS)
print("WEBSITE_ROOT:", WEBSITE_ROOT)

PROJECT_ROOT: .
ARTIFACTS: ./artifacts
WEBSITE_ROOT: ./docs/assets/model_selection


In [2]:
from ips_learning.model_selection import (
    ModelSelectionRepeatedTrialRunConfig,
    ModelSelectionRepeatedTrialEvaluationRunner,
)

SYSTEM_CONFIGS = [
    {
        "system_name": "sppcs",
        "profile_name": "clumps",
        "model_selection_mode": "full_structure_selection",
        "fit_end_time": 0.5,
        "prediction_end_time": 2.0,
    },
    {
        "system_name": "levi",
        "profile_name": "levi_velocity_grid",
        "model_selection_mode": "full_structure_selection",
        "fit_end_time": 0.75,
        "prediction_end_time": 2.25,
    },
    {
        "system_name": "cs",
        "profile_name": "profile1",
       "model_selection_mode": "full_structure_selection",
        "fit_end_time": 0.5,
        "prediction_end_time": 2.0,
    },
    {
        "system_name": "opinion",
        "profile_name": "profile1",
        "model_selection_mode": "full_structure_selection",
        "fit_end_time": 10.0,
        "prediction_end_time": 20.0,
        "radial_n_basis": 50,
    },
]

In [3]:
def export_repeated_trial_tables(*, system_name: str, profile_name: str, output_dir: Path) -> Path:
    """Copy repeated-trial summary CSV/TEX files to docs/assets/model_selection."""
    src_dir = (
        ARTIFACTS
        / "model_selection"
        / system_name
        / profile_name
        / "repeated_trials"
        / "summaries"
    )

    dst_dir = WEBSITE_ROOT / system_name / "repeated_trials"
    dst_dir.mkdir(parents=True, exist_ok=True)

    copied = []
    for path in sorted(src_dir.glob("*")):
        if path.suffix.lower() in {".csv", ".tex", ".json"}:
            dst = dst_dir / path.name
            shutil.copy2(path, dst)
            copied.append(str(dst.relative_to(PROJECT_ROOT)))

    manifest = {
        "artifact_type": "website_repeated_trial_export",
        "system_name": system_name,
        "profile_name": profile_name,
        "source_dir": str(src_dir.relative_to(PROJECT_ROOT)),
        "copied_files": copied,
    }

    with open(dst_dir / "manifest_repeated_trials.json", "w", encoding="utf-8") as f:
        json.dump(manifest, f, indent=2)

    return dst_dir

In [4]:
all_outputs = {}

for cfg_dict in SYSTEM_CONFIGS:
    system_name = cfg_dict["system_name"]
    profile_name = cfg_dict["profile_name"]

    print("\n" + "=" * 80)
    print("Repeated-trial model selection:", system_name, profile_name)
    print("=" * 80)

    rt_config = ModelSelectionRepeatedTrialRunConfig(
        system_name=system_name,
        profile_name=profile_name,
        model_selection_mode=cfg_dict["model_selection_mode"],
        fit_end_time=cfg_dict["fit_end_time"],
        prediction_end_time=cfg_dict["prediction_end_time"],
        num_trials=10,
        trial_start=1,
        radial_n_basis=cfg_dict.get("radial_n_basis", 10),
        environment_n_basis_per_dim=10,
        primary_metric="E_traj_full",
        relative_tie_tol=0.05,
        absolute_tie_tol=1e-3,
        max_condition_number=1e12,
        artifact_root=ARTIFACTS,

        include_noise_variance_parameter=True,
    )

    rt_eval = ModelSelectionRepeatedTrialEvaluationRunner(rt_config)
    rt_outputs = rt_eval.run()

    website_dir = export_repeated_trial_tables(
        system_name=system_name,
        profile_name=profile_name,
        output_dir=WEBSITE_ROOT,
    )

    all_outputs[system_name] = {
        "outputs": rt_outputs,
        "website_dir": website_dir,
    }

    display(rt_outputs["trial_df"])
    display(rt_outputs["summaries"]["selection_frequency"])
    display(
        rt_outputs["summaries"][
            "aic_selection_frequency"
        ]
    )

    display(
        rt_outputs["summaries"][
            "bic_selection_frequency"
        ]
    )

    display(
        rt_outputs["summaries"][
            "method_agreement_summary"
        ]
    )
    display(rt_outputs["summaries"]["order_frequency"])
    display(rt_outputs["summaries"]["interaction_frequency"])
    display(rt_outputs["summaries"]["environment_frequency"])
    display(rt_outputs["summaries"]["selected_metric_summary"])
    display(rt_outputs["summaries"]["candidate_metric_summary"])

    print("Website repeated-trial export:", website_dir)


Repeated-trial model selection: sppcs clumps

Running model selection trial 001

Running model selection trial 002

Running model selection trial 003

Running model selection trial 004

Running model selection trial 005

Running model selection trial 006

Running model selection trial 007

Running model selection trial 008

Running model selection trial 009

Running model selection trial 010
FOUND ./artifacts/model_selection/sppcs/clumps/repeated_trials/summaries/repeated_trial_model_selection_metrics.csv
FOUND ./artifacts/model_selection/sppcs/clumps/repeated_trials/summaries/repeated_trial_model_selection_config.json
FOUND ./artifacts/model_selection/sppcs/clumps/repeated_trials/summaries/selection_frequency.csv
FOUND ./artifacts/model_selection/sppcs/clumps/repeated_trials/summaries/selection_frequency.tex
FOUND ./artifacts/model_selection/sppcs/clumps/repeated_trials/summaries/aic_selection_frequency.csv
FOUND ./artifacts/model_selection/sppcs/clumps/repeated_trials/summaries/aic_

./.venv/lib/python3.13/site-packages/pandas/core/nanops.py:1027: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)


,trial,system_name,profile_name,model_selection_mode,failed,failure_message,validation_dataset,test_dataset,validation_Tf,test_Tf,...,first_order_energy_f__trajectory_reconstruction_rss,first_order_energy_f__trajectory_reconstruction_n_obs,first_order_energy_f__n_model_coefficients,first_order_energy_f__n_likelihood_parameters,first_order_energy_f__AIC,first_order_energy_f__BIC,first_order_energy_f__aic_rank,first_order_energy_f__bic_rank,first_order_energy_f__aic_selected,first_order_energy_f__bic_selected
0,1,sppcs,clumps,full_structure_selection,False,,dataset_validation,dataset_test,2.0,2.0,...,NaN,NaN,210,NaN,NaN,NaN,NaN,NaN,False,False
1,2,sppcs,clumps,full_structure_selection,False,,dataset_validation,dataset_test,2.0,2.0,...,NaN,NaN,210,NaN,NaN,NaN,NaN,NaN,False,False
2,3,sppcs,clumps,full_structure_selection,False,,dataset_validation,dataset_test,2.0,2.0,...,NaN,NaN,210,NaN,NaN,NaN,NaN,NaN,False,False
3,4,sppcs,clumps,full_structure_selection,False,,dataset_validation,dataset_test,2.0,2.0,...,NaN,NaN,210,NaN,NaN,NaN,NaN,NaN,False,False
4,5,sppcs,clumps,full_structure_selection,False,,dataset_validation,dataset_test,2.0,2.0,...,NaN,NaN,210,NaN,NaN,NaN,NaN,NaN,False,False
5,6,sppcs,clumps,full_structure_selection,False,,dataset_validation,dataset_test,2.0,2.0,...,NaN,NaN,210,NaN,NaN,NaN,NaN,NaN,False,False
6,7,sppcs,clumps,full_structure_selection,False,,dataset_validation,dataset_test,2.0,2.0,...,NaN,NaN,210,NaN,NaN,NaN,NaN,NaN,False,False
7,8,sppcs,clumps,full_structure_selection,False,,dataset_validation,dataset_test,2.0,2.0,...,NaN,NaN,210,NaN,NaN,NaN,NaN,NaN,False,False
8,9,sppcs,clumps,full_structure_selection,False,,dataset_validation,dataset_test,2.0,2.0,...,NaN,NaN,210,NaN,NaN,NaN,NaN,NaN,False,False
9,10,sppcs,clumps,full_structure_selection,False,,dataset_validation,dataset_test,2.0,2.0,...,NaN,NaN,210,NaN,NaN,NaN,NaN,NaN,False,False


,selected_candidate,selected_count,selected_fraction,num_valid_trials
0,second_order_energy_alignment_f,10,1.0,10


,aic_selected_candidate,selected_count,selected_fraction,num_valid_trials
0,second_order_energy_alignment_f,10,1.0,10


,bic_selected_candidate,selected_count,selected_fraction,num_valid_trials
0,second_order_energy_alignment_f,10,1.0,10


... (39 outputs left out)

,selected_include_environment,selected_count,selected_fraction,num_valid_trials
0,False,10,1.0,10


,metric,mean,trial_std,mean_pm_trial_std,num_trials,system_name,profile_name,model_selection_mode
0,selected_E_traj_full,0.025690,0.005916,2.5690e-02 ± 5.9159e-03,10,opinion,profile1,full_structure_selection
1,selected_E_traj_full_std,0.019963,0.006334,1.9963e-02 ± 6.3344e-03,10,opinion,profile1,full_structure_selection
2,selected_E_traj_reconstruction,0.048527,0.009608,4.8527e-02 ± 9.6081e-03,10,opinion,profile1,full_structure_selection
3,selected_E_traj_reconstruction_rel,0.007718,0.001810,7.7181e-03 ± 1.8103e-03,10,opinion,profile1,full_structure_selection
4,selected_E_traj_prediction,0.048527,0.009608,4.8527e-02 ± 9.6081e-03,10,opinion,profile1,full_structure_selection
5,selected_E_res,0.005806,0.000587,5.8058e-03 ± 5.8661e-04,10,opinion,profile1,full_structure_selection
6,selected_E_res_rel,0.179927,0.014015,1.7993e-01 ± 1.4015e-02,10,opinion,profile1,full_structure_selection
7,selected_complexity,1.000000,0.000000,1.0000e+00 ± 0.0000e+00,10,opinion,profile1,full_structure_selection
8,selected_phi_E_norm,0.358577,0.017518,3.5858e-01 ± 1.7518e-02,10,opinion,profile1,full_structure_selection
9,selected_phi_A_norm,0.000000,0.000000,0.0000e+00 ± 0.0000e+00,10,opinion,profile1,full_structure_selection


,candidate,num_trials,system_name,profile_name,model_selection_mode,primary_value_mean,primary_value_trial_std,primary_value_mean_pm_trial_std,primary_value_num_trials,E_traj_full_mean,...,BIC_mean_pm_trial_std,BIC_num_trials,aic_rank_mean,aic_rank_trial_std,aic_rank_mean_pm_trial_std,aic_rank_num_trials,bic_rank_mean,bic_rank_trial_std,bic_rank_mean_pm_trial_std,bic_rank_num_trials
0,first_order_energy_f,10,opinion,profile1,full_structure_selection,0.026007,0.006254,2.6007e-02 ± 6.2542e-03,10,0.026007,...,-1.3660e+05 ± 2.1234e+04,10,1.200000,0.421637,1.2000e+00 ± 4.2164e-01,10,1.200000,0.421637,1.2000e+00 ± 4.2164e-01,10
1,first_order_energy_only,10,opinion,profile1,full_structure_selection,0.025690,0.005916,2.5690e-02 ± 5.9159e-03,10,0.025690,...,-1.3619e+05 ± 2.1743e+04,10,1.800000,0.421637,1.8000e+00 ± 4.2164e-01,10,1.800000,0.421637,1.8000e+00 ± 4.2164e-01,10
2,second_order_alignment_f,10,opinion,profile1,full_structure_selection,inf,NaN,nan ± nan,10,0.091120,...,-8.9394e+04 ± 9.0576e+03,8,4.000000,0.000000,4.0000e+00 ± 0.0000e+00,8,4.000000,0.000000,4.0000e+00 ± 0.0000e+00,8
3,second_order_alignment_only,10,opinion,profile1,full_structure_selection,inf,NaN,nan ± nan,10,0.091120,...,-8.9500e+04 ± 9.0570e+03,8,3.000000,0.000000,3.0000e+00 ± 0.0000e+00,8,3.000000,0.000000,3.0000e+00 ± 0.0000e+00,8
4,second_order_energy_alignment_f,10,opinion,profile1,full_structure_selection,inf,NaN,nan ± nan,10,0.332191,...,-3.2674e+04 ± 3.4796e+04,7,6.428571,1.133893,6.4286e+00 ± 1.1339e+00,7,6.428571,1.133893,6.4286e+00 ± 1.1339e+00,7
5,second_order_energy_alignment_only,10,opinion,profile1,full_structure_selection,inf,NaN,nan ± nan,10,0.332189,...,-3.2784e+04 ± 3.4799e+04,7,5.428571,1.133893,5.4286e+00 ± 1.1339e+00,7,5.428571,1.133893,5.4286e+00 ± 1.1339e+00,7
6,second_order_energy_f,10,opinion,profile1,full_structure_selection,inf,NaN,nan ± nan,10,7.321181,...,9.0632e+04 ± 1.2669e+05,9,6.666667,1.802776,6.6667e+00 ± 1.8028e+00,9,6.666667,1.802776,6.6667e+00 ± 1.8028e+00,9
7,second_order_energy_only,10,opinion,profile1,full_structure_selection,0.354304,0.015385,3.5430e-01 ± 1.5385e-02,10,0.354304,...,-4.5815e+04 ± 9.8986e+03,10,5.200000,0.788811,5.2000e+00 ± 7.8881e-01,10,5.200000,0.788811,5.2000e+00 ± 7.8881e-01,10


Website repeated-trial export: ./docs/assets/model_selection/opinion/repeated_trials


In [5]:
# # Compact final overview
# overview_rows = []

# for system_name, item in all_outputs.items():
#     freq = item["outputs"]["summaries"]["selection_frequency"].copy()
#     if not freq.empty:
#         freq["system_name"] = system_name
#         overview_rows.append(freq)

# if overview_rows:
#     repeated_selection_overview = pd.concat(overview_rows, ignore_index=True)
#     display(repeated_selection_overview)
# else:
#     repeated_selection_overview = pd.DataFrame()
#     print("No repeated-trial selection-frequency rows were produced.")

In [6]:
# ============================================================
# Compact comparison of proposed method, AIC, and BIC
# ============================================================

TRUE_CANDIDATES = {
    "sppcs": "second_order_energy_alignment_f",
    "levi": "second_order_alignment_f",
    "cs": "second_order_alignment_only",
    "opinion": "first_order_energy_only",
}

method_specs = [
    (
        "Proposed",
        "selection_frequency",
        "selected_candidate",
    ),
    (
        "AIC",
        "aic_selection_frequency",
        "aic_selected_candidate",
    ),
    (
        "BIC",
        "bic_selection_frequency",
        "bic_selected_candidate",
    ),
]

overview_rows = []

for system_name, item in all_outputs.items():
    summaries = item["outputs"]["summaries"]

    for method, summary_name, candidate_column in method_specs:
        frequency_df = summaries[summary_name].copy()

        if frequency_df.empty:
            continue

        frequency_df = frequency_df.rename(
            columns={
                candidate_column: "selected_candidate"
            }
        )

        frequency_df["method"] = method
        frequency_df["system_name"] = system_name

        frequency_df["true_candidate"] = (
            TRUE_CANDIDATES.get(system_name)
        )

        frequency_df["correct_structure"] = (
            frequency_df["selected_candidate"]
            == frequency_df["true_candidate"]
        )

        overview_rows.append(frequency_df)

if overview_rows:
    repeated_selection_overview = pd.concat(
        overview_rows,
        ignore_index=True,
    )

    display(
        repeated_selection_overview[
            [
                "system_name",
                "method",
                "selected_candidate",
                "selected_count",
                "selected_fraction",
                "num_valid_trials",
                "true_candidate",
                "correct_structure",
            ]
        ]
    )

    exact_recovery_summary = (
        repeated_selection_overview.loc[
            repeated_selection_overview[
                "correct_structure"
            ]
        ][
            [
                "system_name",
                "method",
                "selected_fraction",
                "num_valid_trials",
            ]
        ]
        .rename(
            columns={
                "selected_fraction":
                    "exact_structure_recovery_rate"
            }
        )
        .sort_values(
            ["system_name", "method"]
        )
        .reset_index(drop=True)
    )

    display(exact_recovery_summary)

else:
    repeated_selection_overview = pd.DataFrame()
    exact_recovery_summary = pd.DataFrame()

    print(
        "No repeated-trial selection-frequency "
        "rows were produced."
    )

,system_name,method,selected_candidate,selected_count,selected_fraction,num_valid_trials,true_candidate,correct_structure
0,sppcs,Proposed,second_order_energy_alignment_f,10,1.0,10,second_order_energy_alignment_f,True
1,sppcs,AIC,second_order_energy_alignment_f,10,1.0,10,second_order_energy_alignment_f,True
2,sppcs,BIC,second_order_energy_alignment_f,10,1.0,10,second_order_energy_alignment_f,True
3,levi,Proposed,second_order_alignment_f,10,1.0,10,second_order_alignment_f,True
4,levi,AIC,second_order_alignment_f,9,0.9,10,second_order_alignment_f,True
5,levi,AIC,second_order_energy_alignment_f,1,0.1,10,second_order_alignment_f,False
6,levi,BIC,second_order_alignment_f,9,0.9,10,second_order_alignment_f,True
7,levi,BIC,second_order_energy_alignment_f,1,0.1,10,second_order_alignment_f,False
8,cs,Proposed,second_order_alignment_only,10,1.0,10,second_order_alignment_only,True
9,cs,AIC,second_order_energy_alignment_f,6,0.6,10,second_order_alignment_only,False


,system_name,method,exact_structure_recovery_rate,num_valid_trials
0,cs,Proposed,1.0,10
1,levi,AIC,0.9,10
2,levi,BIC,0.9,10
3,levi,Proposed,1.0,10
4,opinion,AIC,0.2,10
5,opinion,BIC,0.2,10
6,opinion,Proposed,1.0,10
7,sppcs,AIC,1.0,10
8,sppcs,BIC,1.0,10
9,sppcs,Proposed,1.0,10
